# Analyzing a Neo4j Graph Database via Cypher + GDS
## Assignment 2 - Approach

**Course:** Spring 2026 Capstone  
**Assignment:** Analyzing a Neo4j Graph Database via Cypher + GDS  
**Student:** Bekithemba Nkomo  


---

## 1. Problem Statement

This assignment requires analyzing a pre-built Neo4j graph database derived from the FDA's Adverse Event Reporting System (FAERS), a real-world healthcare surveillance database that tracks adverse drug reactions reported by physicians, pharmacists, nurses, patients, and manufacturers.

We are given an existing graph and must first discover its structure through exploratory data analysis (EDA) before answering increasingly complex analytical questions. This mirrors how a real data scientist operates: rarely do you receive a perfectly documented dataset. Instead, you must interrogate the data itself to understand what it contains before deriving insight from it.

The assignment has three tiers of increasing complexity:

1. **Structural EDA:** Understanding the graph like what nodes, relationships, and properties exist
2. **Domain Analytics:** Answering of clinical questions about drugs, reactions, outcomes, and manufacturers
3. **GDS Graph Algorithms**: Applying node similarity and community detection to uncover patient risk patterns

---

## 2. Thinking About the Domain First

Before writing a single query, we want to reason about what this graph represents, because understanding the domain shapes every analytical decision.

### What is FAERS?

FAERS is a voluntary adverse event reporting system. When a patient takes a drug and experiences a negative reaction, that event can be reported by a healthcare professional, consumer, or manufacturer. Each report becomes a **Case** in the system.

This immediately tells us several important things about the data:

- **It is inherently relational:** A Case connects a Patient to one or more Drugs, Reactions, Outcomes, and Reporters, this is exactly the kind of multi-entity, relationship-rich data that graph databases excel at representing
- **It is voluntary and self-reported:** This means there will be biases, gaps, and inconsistencies in the data. Some reactions may be over-reported for example common side effects, while rare but serious events may be under-reported
- **It is clinically significant:** The end goal of FAERS analysis is patient safety, identifying drugs with disproportionate harm, finding at-risk patient populations, and detecting emerging safety signals

### Thinking About the Eight Node Types

The assignment tells us the graph has eight node types. Rather than treating these as abstract labels, we want to think about what each one *is* in the real world:

| Node Type | What It Represents | Key Question It Enables |
|-----------|-------------------|------------------------|
| **Case** | A single adverse event report thus the central entity | Who experienced this event? What were their demographics? |
| **Drug** | A drug that may have caused the adverse event | Which drugs are most dangerous? What suspect types exist? |
| **Reaction** | The specific adverse reaction experienced | What are the most common reactions? Which are most severe? |
| **Outcome** | Long-term result of the adverse event for example Death, Disability | How serious was the harm? |
| **Manufacturer** | The pharmaceutical company that makes the drug | Which companies have the worst safety profiles? |
| **Therapy** | The treatment regimen the patient was on | Was the reaction related to a specific therapy? |
| **ReportSource** | Who filed the report (doctor, patient, manufacturer) | How reliable is the report? Do sources differ in what they report? |
| **AgeGroup** | Patient age demographic category | Are certain age groups more vulnerable to specific drug reactions? |

Thinking this way, as a domain expert, not just a database engineer allows us to anticipate what the relationships between these nodes will look like and what the most insightful queries will be.

### Thinking About Relationships

The assignment notes there are "many different types of relationships." Based on our domain understanding, we expect to find relationships such as:

- `(Case)-[:HAS_REACTION]->(Reaction)` - what the patient experienced
- `(Case)-[:RESULTED_IN]->(Outcome)` - what happened to the patient long-term
- `(Case)-[:REPORTED_BY]->(ReportSource)` - who filed the report
- `(Case)-[:IS_PRIMARY_SUSPECT]->(Drug)` or similar suspect-type relationships - the assignment explicitly notes that the Case-Drug relationship encodes the drug's suspect type
- `(Case)-[:BELONGS_TO_AGE_GROUP]->(AgeGroup)` - patient demographics
- `(Drug)-[:MANUFACTURED_BY]->(Manufacturer)` - drug provenance
- `(Drug)-[:IN_THERAPY]->(Therapy)` - treatment context

**Critically:** The assignment notes the suspect type (primary suspect, secondary suspect, concomitant, interacting) is encoded in the *relationship* between Case and Drug , not as a property on the Drug node. This is a graph modeling insight that will directly affect how we write queries for the analytical section.

---

## 3. Planned Approach

### Phase 1: Structural EDA - Discovering the Graph

Since no schema documentation is provided, my first task is to interrogate the graph itself. I will treat this like a detective investigation: start broad, then progressively narrow in.

**Step 1 - Count everything:** Total node count, total relationship count. This establishes the scale of the graph and helps interpret query performance later.

**Step 2 - Discover all labels and relationship types:** These are the 'vocabulary' of the graph. we must know all node labels and all relationship type names before we can write meaningful analytical queries.

**Step 3 - Discover properties per node type:** Each node type has different properties. We will write a query that, for each distinct label, returns the set of unique property keys. This tells us what we can filter, sort, and aggregate on.

**Step 4 - Explore key properties:** The assignment asks specifically about gender values on Case nodes and distinct Reaction descriptions. These are entry points into the demographic and clinical content of the graph.

**Step 5 - Understand Outcome descriptions:** Outcome is a critical node type for the analytical section. we need to know the exact string values used for example is it 'Death' or 'Died' or 'Fatal'?, so our downstream queries match correctly.

### Phase 2: Domain Analytics - Answering Clinical Questions

With schema knowledge in hand, we will construct four analytical queries. Our approach for each:

**Query 1 -  Most frequent reactions:**
This is a frequency analysis across all Case-Reaction relationships. We will group by reaction name, count occurrences, and sort descending. The key question to ask before writing this query: is frequency counted per Case, or per unique drug? We will count per Case (patient-level frequency) as that is most clinically meaningful.

**Query 2 - Drugs producing most severe outcomes:**
This requires traversing from Drug → Case → Outcome and filtering for the four severe outcome types. A critical design decision: should we count Cases (patients) or unique Drug-Outcome combinations? We will count distinct Cases to measure patient impact. We will also consider whether to include all suspect types or only primary suspects, a clinically important distinction.

**Query 3 - Manufacturers with most drugs with side effects:**
This traverses Manufacturer → Drug → Case → Reaction. We need to count distinct drugs per manufacturer, not reactions, the question asks about the breadth of their drug portfolio with issues, not the volume of reports.

**Query 4 - Top drugs and side effects for the worst manufacturer:**
This is a nested analytical query, first find the top manufacturer, then for that specific manufacturer, find their top 5 drugs and each drug's distinct side effects. We will use a subquery or WITH clause to chain these.

### Phase 3: GDS Algorithms - Graph-Native Analysis

This is the most sophisticated section and requires careful algorithm selection:

**GDS Question 1 - Node Similarity for Patient Journeys:**

The question asks to identify patients who experienced identical sequences of reactions after taking the same drugs. This is asking: which patients are most similar to each other based on their drug-reaction profiles?

The key word here is **Jaccard Similarity** - the most appropriate algorithm from GDS for this use case. It measures the overlap between two sets (in this case, the set of Drug+Reaction combinations for each patient). Two patients with high Jaccard similarity had highly overlapping drug-reaction experiences.

**Why Jaccard over other similarity algorithms?**
- **Jaccard** measures set intersection or set union - ideal when we care about which items appear, not how many times they appear
- **Cosine similarity** would over-weight patients with many reactions simply for having more entries
- **Overlap coefficient** would bias toward patients with fewer reactions
- Jaccard treats each drug-reaction pair as a binary feature: either the patient experienced it or they didn't

For the graph projection, we will project Cases and their connected Drugs and Reactions, treating the combination of drug + reaction as shared 'neighbors.'

**GDS Question 2 - Community Detection for Patient Sub-Phenotypes:**

The question asks to cluster patients based on shared demographics AND shared adverse reactions, looking for patient sub-phenotypes who may be at higher risk for specific drug-reaction combinations.

The most appropriate algorithm here is **Louvain Community Detection** (or alternatively **Label Propagation**). Reasons:

- **Louvain** optimizes modularity - it finds dense clusters of nodes that are more connected to each other than to the rest of the graph. Patients who share demographic characteristics (age group, gender) AND share the same reactions will naturally form dense subgraphs.
- **Why not K-Means?** K-Means is not graph-native and doesn't leverage the relational structure we have built.
- **Why not PageRank?** PageRank measures influence/centrality, not community membership.
- **Why not Betweenness Centrality?** That identifies bridge nodes, not clusters.

For the graph projection, we will project Cases connected to both AgeGroup nodes (demographic features) and Reaction nodes (clinical features), creating a bipartite-style projection where Cases sharing both types of neighbors will form communities.

---

## 4. Anticipated Challenges

### Challenge 1: Undocumented Schema
We do not know the exact property names, relationship types, or string values used in this database. Every analytical query depends on knowing the exact schema. We must complete Phase 1 (EDA) thoroughly before Phase 2, or our queries will silently return no results.

**Mitigation:** We will build EDA queries that are defensive, using `CALL apoc.meta.schema()` if available, or writing label-agnostic property inspection queries. We will validate every assumption about property names and values before using them in analytical queries.

### Challenge 2: Relationship Type Encoding of Drug Suspect Types
The assignment explicitly states that the suspect type (primary, secondary, concomitant, interacting) is encoded in the relationship between Case and Drug, not as a drug property. This means there may be *multiple different relationship types* connecting Case to Drug nodes.

**Mitigation:** Our EDA will specifically inspect all relationship types between Case and Drug nodes. Analytical queries about drug severity will need to handle this carefully, for the strictest analysis we should filter to primary suspects only; for broader analysis we may include all suspect types.

### Challenge 3: Data Quality in a Voluntary Reporting System
FAERS is voluntary - meaning reports are biased toward severe events, well-known drugs, and reports filed by professionals. Frequency of reactions does not equal population-level risk. A drug appearing frequently may simply be more widely prescribed.

**Mitigation:** We will acknowledge these limitations in our analytical commentary. For example, when reporting the 'top drugs producing severe outcomes', We will note that frequency may reflect market share as much as true risk.

### Challenge 4: GDS Graph Projections
GDS algorithms require creating an in-memory graph projection. The projection must include the right nodes and relationships, and the graph must be appropriately configured for example, for undirected vs. directed, for node properties to be used as weights.

**Mitigation:** We will first inspect the graph structure thoroughly in Phase 1. For Jaccard similarity, we need to ensure the projection represents Cases as nodes with shared neighbors (Drugs + Reactions). For Louvain, we need to create a monopartite projection (Case-to-Case similarity) or use a bipartite approach with proper relationship weighting.

### Challenge 5: Scale and Query Performance
The FAERS database is a substantial real-world dataset. Complex traversals for example, Manufacturer → Drug → Case → Reaction could be slow without proper query optimization.

**Mitigation:** We will use `LIMIT` clauses during exploration, leverage any existing indexes, and write queries that anchor on the most selective conditions first for example start from a specific Manufacturer rather than scanning all Cases.

### Challenge 6: GDS Memory for Large Graph Projections
Projecting the full graph into memory for node similarity could be expensive. With thousands of Cases each connected to many Reactions and Drugs, the similarity computation is O(n²).

**Mitigation:** We will use `topK` and `similarityCutoff` parameters in the GDS Jaccard query to limit the output to the most meaningful similar pairs. We will also consider projecting only a semantically relevant subgraph rather than the entire database.

---

## 5. Rationale for this Approach

Rather than simply restating the assignment requirements, this approach reflects the following deeper commitments:

**1. Domain-first thinking:** We reasoned about what each node type *represents* in healthcare before deciding how to query it. This prevents the common mistake of writing syntactically correct queries that are clinically meaningless.

**2. Algorithm justification grounded in data characteristics:** Rather than picking GDS algorithms arbitrarily, we evaluated each option against the specific structure of this data, Jaccard for set-based patient similarity, Louvain for modularity-based community detection. Each choice is defensible.

**3. Acknowledging data limitations upfront:** A real data scientist communicates the boundaries of their analysis. FAERS is a voluntary reporting system, and conclusions drawn from it must be appropriately hedged.

**4. Dependency-aware execution plan:** We identified that Phase 1 (EDA) is a prerequisite for Phase 2 (Analytics), not just sequentially, but because specific string values discovered in EDA (like exact Outcome descriptions) directly feed into Phase 2 filter conditions.

**5. Thinking about the relationship encoding of suspect type:** The assignment notes this is encoded in the Case-Drug relationship type, a subtle but important detail that most analyses would miss. We have flagged this as a key consideration in query design.

---

## 6. Execution Plan Summary

| Phase | Task | Key Dependency |
|-------|------|----------------|
| EDA 1-4 | Count nodes, labels, relationships, relationship types | None - run first |
| EDA 5 | Discover properties per node type | Labels known from EDA 2 |
| EDA 6-8 | Explore gender, reactions, outcome values | Property names known from EDA 5 |
| Analytics 1 | Top 20 most frequent reactions | Reaction property name from EDA |
| Analytics 2 | Drugs with most severe outcomes | Outcome values from EDA 8 + relationship types from EDA 4 |
| Analytics 3 | Manufacturers with most drugs with side effects | All node types confirmed via EDA |
| Analytics 4 | Top drugs + side effects for worst manufacturer | Result of Analytics 3 |
| GDS 1 | Jaccard node similarity - patient journeys | Graph structure fully understood |
| GDS 2 | Louvain community detection - patient sub-phenotypes | Demographics + reactions confirmed in EDA |

---

## 7. References
- https://graphacademy.neo4j.com/courses/gds-fundamentals/

- https://graphacademy.neo4j.com/courses/gds-shortest-paths/?category=analytics

- https://youtu.be/zMn3_Yq_eGg - Graph Data Science Meetup

